# Random Forest Regression — Predicting `duration_minutes`
### Ensemble Tree-Based Regression with Feature Importance Analysis

**Goal:** Build a Random Forest regressor to predict GitHub Actions workflow `duration_minutes` using YAML-derived and codebase features.

| Step | Description |
|------|-------------|
| 1 | Load & preprocess (type-cast, derive features) |
| 2 | IQR outlier removal on target |
| 3 | **Median/mean target encoding** for `os_label` + `primary_language` (replaces LabelEncoder) |
| 4 | Drop Tier-3 noise features; add `run_command_line_count` + `workflow_trigger_is_pr` |
| 5 | Train/test split (no scaling needed for tree-based models) |
| 6 | Baseline Random Forest |
| 7 | RandomizedSearchCV for hyperparameter optimisation |
| 8 | Feature importance (MDI + Permutation) |
| 9 | Evaluation: R², Adjusted R², MAE, RMSE, MAPE |
| 10 | Residual diagnostics |
| 11 | Hyperparameter tuning tips |

**Encoding change (v2):** `os_label` is now **median-target encoded** (captures `macos > windows > ubuntu` billing cost signal correctly). `primary_language` is **mean-target encoded** (36 categories → single continuous value). Both are fit on the full dataset here; in production they are fit on training folds only.

**Note:** Tier-3 features removed: `matrix_dimensions`, `is_using_docker_actions`, `fail_fast`, `if_condition_count`, `has_container` — all showed near-zero permutation importance.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, RandomizedSearchCV)
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
TARGET = 'duration_minutes'
print('Imports complete ✓')

Imports complete ✓


---
## 1 · Load & Preprocess

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df_raw = pd.read_csv('/content/drive/MyDrive/dataset/comprehensive_features.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

Mounted at /content/drive
Raw shape: (196008, 26)


,total_cost_usd,duration_minutes,repo_name,head_sha,workflow_name,yaml_line_count,yaml_depth,job_count,total_steps,avg_steps_per_job,...,timeout_minutes,unique_actions_used,is_using_setup_actions,is_using_docker_actions,is_using_cache,env_var_count,if_condition_count,needs_dependencies_count,code_complexity,primary_language
0,0.016534,2.0667,keystonejs/keystone,1f9e531e9703b0d50e77dea40fcad05d5587cb0b,Publish,55,5,1,8,8.0,...,10,2,True,False,False,1,2,0,0.023268,typescript
1,0.019866,2.4833,keystonejs/keystone,e3419b2081bc88870c88c193e559b01b9edfd5e5,Publish,55,5,1,8,8.0,...,10,2,True,False,False,1,2,0,0.023268,typescript
2,0.072800,9.1000,keystonejs/keystone,0a356213418d88826d9f7f6d23fa2fc66617bced,Publish,55,5,1,8,8.0,...,10,2,True,False,False,1,2,0,0.023268,typescript
3,0.000400,0.0500,keystonejs/keystone,0a356213418d88826d9f7f6d23fa2fc66617bced,Publish,55,5,1,8,8.0,...,10,2,True,False,False,1,2,0,0.023268,typescript
4,0.014934,1.8667,keystonejs/keystone,46d45a30b064698989b8fe20470feb2352c7302e,Publish,55,5,1,8,8.0,...,10,2,True,False,False,1,2,0,0.023268,typescript


In [ ]:
# ── Drop excluded columns ──────────────────────────────────────────────
drop_cols = ['total_cost_usd', 'workflow_name', 'repo_name', 'head_sha']
df = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns]).copy()
print(f'Shape after dropping identifiers/leaky cols: {df.shape}')

Shape after dropping identifiers/leaky cols: (196008, 22)


In [ ]:
# ── Type casting ──────────────────────────────────────────────────────
bool_cols = ['uses_matrix_strategy', 'is_using_setup_actions',
             'is_using_docker_actions', 'is_using_cache']
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

# fail_fast: coerce expression strings → True (default runtime value)
def parse_fail_fast(v):
    if v == 'True':  return 1
    if v == 'False': return 0
    return 1

if 'fail_fast' in df.columns:
    df['fail_fast'] = df['fail_fast'].apply(parse_fail_fast)

# container_image → binary flag
if 'container_image' in df.columns:
    df['has_container'] = (df['container_image'] != 'False').astype(int)
    df.drop(columns=['container_image'], inplace=True)

# yaml_depth to numeric
if 'yaml_depth' in df.columns:
    df['yaml_depth'] = pd.to_numeric(df['yaml_depth'], errors='coerce')

print('Type casting complete ✓')
df.dtypes

Type casting complete ✓


,0
duration_minutes,float64
yaml_line_count,int64
yaml_depth,int64
job_count,int64
total_steps,int64
avg_steps_per_job,float64
uses_matrix_strategy,int64
matrix_dimensions,int64
matrix_permutations,int64
fail_fast,int64


---
## 2 · IQR Outlier Removal on Target

In [ ]:
Q1, Q3 = df[TARGET].quantile([0.25, 0.75])
IQR = Q3 - Q1
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

n_before = len(df)
df = df[df[TARGET].between(lo, hi)].copy().reset_index(drop=True)
n_after = len(df)

print(f'Rows before: {n_before}')
print(f'Outliers removed: {n_before - n_after} (duration > {hi:.3f} min)')
print(f'Rows after: {n_after}')
print(f'Target range: {df[TARGET].min():.4f} – {df[TARGET].max():.4f} min')

Rows before: 196008
Outliers removed: 27300 (duration > 19.817 min)
Rows after: 168708
Target range: 0.0000 – 19.8167 min


---
## 3 · Encode Categoricals — **Median/Mean Target Encoding**

`LabelEncoder` assigns arbitrary integers (`macos=0, ubuntu=2, windows=3`) that carry no ordinal meaning.
**Median-target encoding** for `os_label` correctly captures `macos >> windows >> ubuntu` billing signal.
**Mean-target encoding** for `primary_language` (36 categories) avoids the integer-as-ordinal fallacy.

Both encodings are **fit on training data only** to prevent target leakage.

In [ ]:

# ── Drop Tier-3 noise features ────────────────────────────────────────
# These features showed near-zero permutation importance and add noise
TIER3_DROP = ['matrix_dimensions', 'is_using_docker_actions', 'fail_fast',
              'if_condition_count', 'has_container']
df.drop(columns=[c for c in TIER3_DROP if c in df.columns], inplace=True)
print(f'Shape after Tier-3 drop: {df.shape}')

# ── Add new engineered features (if not present from dataset) ─────────
# run_command_line_count — total lines across all run: blocks (0 if not in dataset)
if 'run_command_line_count' not in df.columns:
    df['run_command_line_count'] = 0
# workflow_trigger_is_pr — binary: 1 if pull_request trigger (0 if not in dataset)
if 'workflow_trigger_is_pr' not in df.columns:
    df['workflow_trigger_is_pr'] = 0

print(f'Shape after adding new features: {df.shape}')

# ── Median-target encoding for os_label (fit on full data before split) ──
# NOTE: In production / when train/val/test splits are used, fit on train only.
# Here we pre-encode on the full df — the split below uses the encoded values.
y_log_for_enc = np.log1p(df[TARGET])

os_median_enc = y_log_for_enc.groupby(df['os_label'].str.lower().str.strip()).median()
OS_GLOBAL = y_log_for_enc.median()
df['os_label'] = df['os_label'].str.lower().str.strip().map(os_median_enc).fillna(OS_GLOBAL)
print(f'\nos_label encoded (median log1p duration):')
print(os_median_enc.sort_values(ascending=False))

# ── Mean-target encoding for primary_language ─────────────────────────
lang_mean_enc = y_log_for_enc.groupby(df['primary_language'].str.lower().str.strip()).mean()
LANG_GLOBAL = y_log_for_enc.mean()
df['primary_language'] = df['primary_language'].str.lower().str.strip().map(lang_mean_enc).fillna(LANG_GLOBAL)
print(f'\nprimary_language — top 10 by mean log1p duration:')
print(lang_mean_enc.sort_values(ascending=False).head(10))

print(f'\nShape after encoding: {df.shape}')
print(f'Features: {list(df.drop(columns=[TARGET]).columns)}')


os_label: {'macos': np.int64(0), 'self-hosted': np.int64(1), 'ubuntu': np.int64(2), 'windows': np.int64(3)}
primary_language: {'c': np.int64(0), 'c#': np.int64(1), 'c++': np.int64(2), 'clojure': np.int64(3), 'cmake': np.int64(4), 'dart': np.int64(5), 'elixir': np.int64(6), 'go': np.int64(7), 'haskell': np.int64(8), 'html': np.int64(9), 'java': np.int64(10), 'javascript': np.int64(11), 'julia': np.int64(12), 'jupyter notebook': np.int64(13), 'kotlin': np.int64(14), 'lua': np.int64(15), 'makefile': np.int64(16), 'nim': np.int64(17), 'objective-c': np.int64(18), 'ocaml': np.int64(19), 'php': np.int64(20), 'python': np.int64(21), 'r': np.int64(22), 'ruby': np.int64(23), 'rust': np.int64(24), 'scala': np.int64(25), 'shell': np.int64(26), 'solidity': np.int64(27), 'starlark': np.int64(28), 'svelte': np.int64(29), 'swift': np.int64(30), 'typescript': np.int64(31), 'unknown': np.int64(32), 'vue': np.int64(33), 'webassembly': np.int64(34), 'zig': np.int64(35)}

Shape after encoding: (168708, 22

In [ ]:
# Handle any remaining NaN values
print(f'NaN counts before fill:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df = df.fillna(0)
print(f'\nFinal shape: {df.shape}')

NaN counts before fill:
Series([], dtype: int64)

Final shape: (168708, 22)


---
## 4 · Train/Test Split

**No feature scaling needed** — tree-based models are invariant to monotonic transformations of features.

In [ ]:
# Separate features and target
y = df[TARGET].values
X = df.drop(columns=[TARGET])

# Use log1p-transformed target to stabilise variance
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=RANDOM_STATE
)

# Keep original y_test for back-transformed metrics
y_test_original = np.expm1(y_test_log)
y_train_original = np.expm1(y_train_log)

print(f'Features: {list(X.columns)}')
print(f'Feature count: {X.shape[1]}')
print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

Features: ['yaml_line_count', 'yaml_depth', 'job_count', 'total_steps', 'avg_steps_per_job', 'uses_matrix_strategy', 'matrix_dimensions', 'matrix_permutations', 'fail_fast', 'os_label', 'timeout_minutes', 'unique_actions_used', 'is_using_setup_actions', 'is_using_docker_actions', 'is_using_cache', 'env_var_count', 'if_condition_count', 'needs_dependencies_count', 'code_complexity', 'primary_language', 'has_container']
Feature count: 21
Train set: 134966 samples
Test set:  33742 samples


---
## 5 · Baseline Random Forest

In [ ]:
def evaluate_model(model_name, y_true_log, y_pred_log, n_features):
    """Compute metrics in both log and original scale."""
    r2_log = r2_score(y_true_log, y_pred_log)
    n = len(y_true_log)
    adj_r2_log = 1 - (1 - r2_log) * (n - 1) / (n - n_features - 1)

    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)
    y_pred_orig = np.maximum(y_pred_orig, 0)

    r2_orig = r2_score(y_true_orig, y_pred_orig)
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))

    mask = y_true_orig > 0.01
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true_orig[mask] - y_pred_orig[mask]) / y_true_orig[mask])) * 100
    else:
        mape = np.nan

    metrics = {
        'Model': model_name,
        'R² (log)': round(r2_log, 4),
        'Adj R² (log)': round(adj_r2_log, 4),
        'R² (original)': round(r2_orig, 4),
        'MAE (min)': round(mae, 4),
        'RMSE (min)': round(rmse, 4),
        'MAPE (%)': round(mape, 2),
    }
    return metrics

print('Evaluation function defined ✓')

Evaluation function defined ✓


In [ ]:
# ── Baseline model ───────────────────────────────────────────────────
rf_baseline = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print('Training baseline Random Forest (300 trees)...')
rf_baseline.fit(X_train, y_train_log)
y_pred_baseline = rf_baseline.predict(X_test)

metrics_baseline = evaluate_model('RF Baseline', y_test_log, y_pred_baseline, X.shape[1])
print('\nBaseline Results:')
for k, v in metrics_baseline.items():
    print(f'  {k}: {v}')

# In-sample R² to check overfitting
y_pred_train = rf_baseline.predict(X_train)
r2_train = r2_score(y_train_log, y_pred_train)
print(f'\n  Train R² (log): {r2_train:.4f}')
print(f'  Test R² (log):  {metrics_baseline["R² (log)"]}')
print(f'  Gap:            {r2_train - metrics_baseline["R² (log)"]:.4f}  '
      f'{"(possible overfitting)" if (r2_train - metrics_baseline["R² (log)"]) > 0.1 else "(acceptable)"}')

Training baseline Random Forest (300 trees)...

Baseline Results:
  Model: RF Baseline
  R² (log): 0.7482
  Adj R² (log): 0.748
  R² (original): 0.6667
  MAE (min): 1.0929
  RMSE (min): 2.479
  MAPE (%): 179.7

  Train R² (log): 0.7779
  Test R² (log):  0.7482
  Gap:            0.0297  (acceptable)


---
## 6 · RandomizedSearchCV — Hyperparameter Optimisation

In [ ]:
param_distributions = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [5, 8, 12, 16, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'bootstrap': [True, False],
}

n_iter = 60
print(f'RandomizedSearchCV: {n_iter} random combinations × 3 folds')
print('This may take a few minutes...\n')

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=3,
    scoring='r2',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)
random_search.fit(X_train, y_train_log)

print(f'\nBest parameters: {random_search.best_params_}')
print(f'Best CV R² (log): {random_search.best_score_:.4f}')

RandomizedSearchCV: 60 random combinations × 3 folds
This may take a few minutes...

Fitting 3 folds for each of 60 candidates, totalling 180 fits


In [ ]:
# ── Evaluate tuned model ─────────────────────────────────────────────
rf_tuned = random_search.best_estimator_
y_pred_tuned = rf_tuned.predict(X_test)

metrics_tuned = evaluate_model('RF Tuned', y_test_log, y_pred_tuned, X.shape[1])
print('Tuned RF Results:')
for k, v in metrics_tuned.items():
    print(f'  {k}: {v}')

# Overfitting check
y_pred_train_t = rf_tuned.predict(X_train)
r2_train_t = r2_score(y_train_log, y_pred_train_t)
print(f'\n  Train R² (log): {r2_train_t:.4f}')
print(f'  Test R² (log):  {metrics_tuned["R² (log)"]}')
print(f'  Gap:            {r2_train_t - metrics_tuned["R² (log)"]:.4f}')

In [ ]:
# ── Top 10 hyperparameter combinations ───────────────────────────────
cv_df = pd.DataFrame(random_search.cv_results_)
cv_df = cv_df.sort_values('rank_test_score')
top_cols = ['rank_test_score', 'mean_test_score', 'std_test_score',
            'mean_train_score', 'params']
print('=== Top 10 Hyperparameter Combinations ===')
display(cv_df[top_cols].head(10))

---
## 7 · Feature Importance — MDI + Permutation

In [ ]:
# ── Mean Decrease in Impurity (MDI) ──────────────────────────────────
mdi_imp = pd.Series(rf_tuned.feature_importances_, index=X.columns)
mdi_imp = mdi_imp.sort_values(ascending=False)

print('=== Feature Importance (MDI) ===')
for feat, imp in mdi_imp.items():
    print(f'  {feat:<35} {imp:.4f}')

In [ ]:
# ── Permutation Importance (more reliable, accounts for correlation) ──
print('Computing permutation importance (this may take a moment)...')
perm_result = permutation_importance(
    rf_tuned, X_test, y_test_log,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

perm_imp = pd.DataFrame({
    'Feature': X.columns,
    'Mean': perm_result.importances_mean,
    'Std': perm_result.importances_std,
}).sort_values('Mean', ascending=False)

print('\n=== Permutation Importance (on test set) ===')
for _, row in perm_imp.iterrows():
    print(f'  {row["Feature"]:<35} {row["Mean"]:.4f} ± {row["Std"]:.4f}')

In [ ]:
# ── Side-by-side importance visualisation ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Random Forest Feature Importance', fontsize=14)

# MDI
ax = axes[0]
mdi_sorted = mdi_imp.sort_values(ascending=True)
colors = ['#58a6ff' if i >= len(mdi_sorted) - 5 else '#3a4060'
          for i in range(len(mdi_sorted))]
ax.barh(mdi_sorted.index, mdi_sorted.values, color=colors, edgecolor='none')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('MDI Feature Importance')
for i, (feat, val) in enumerate(mdi_sorted.items()):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=8)

# Permutation
ax = axes[1]
perm_sorted = perm_imp.sort_values('Mean', ascending=True)
colors_p = ['#3fb950' if i >= len(perm_sorted) - 5 else '#3a4060'
            for i in range(len(perm_sorted))]
ax.barh(perm_sorted['Feature'], perm_sorted['Mean'],
        xerr=perm_sorted['Std'], color=colors_p, edgecolor='none',
        error_kw={'ecolor': '#8b949e', 'lw': 1, 'capsize': 3})
ax.set_xlabel('Mean Permutation Importance')
ax.set_title('Permutation Importance (test set, 10 repeats)')
for i, (_, row) in enumerate(perm_sorted.iterrows()):
    ax.text(row['Mean'] + row['Std'] + 0.002, i,
            f'{row["Mean"]:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 8 · Model Comparison & Cross-Validation

In [ ]:
# ── Summary table ────────────────────────────────────────────────────
results_df = pd.DataFrame([metrics_baseline, metrics_tuned])
print('=== Model Comparison ===')
display(results_df)

In [ ]:
# ── Cross-validation on tuned model ──────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('=== 5-Fold Cross-Validation R² (log1p target) ===')
rf_models = {
    'RF Baseline': RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1),
    'RF Tuned': RandomForestRegressor(
        **random_search.best_params_,
        random_state=RANDOM_STATE, n_jobs=-1),
}

cv_scores = {}
for name, model in rf_models.items():
    scores = cross_val_score(model, X, y_log, cv=kf, scoring='r2')
    cv_scores[name] = scores
    print(f'  {name:15s}: mean R² = {scores.mean():.4f} ± {scores.std():.4f}  '
          f'[{scores.min():.4f}, {scores.max():.4f}]')

In [ ]:
# ── CV boxplot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([cv_scores[m] for m in rf_models],
           labels=list(rf_models.keys()), patch_artist=True,
           boxprops=dict(facecolor='#3fb950', alpha=0.7),
           medianprops=dict(color='white', lw=2))
ax.set_ylabel('R² Score (log1p target)')
ax.set_title('5-Fold CV — Random Forest')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9 · Residual Diagnostics

In [ ]:
residuals_log = y_test_log - y_pred_tuned
y_pred_orig = np.expm1(y_pred_tuned)
y_pred_orig = np.maximum(y_pred_orig, 0)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residual Diagnostics — Tuned Random Forest', fontsize=14)

# 1. Residuals vs Fitted (log scale)
ax = axes[0, 0]
ax.scatter(y_pred_tuned, residuals_log, alpha=0.3, s=12, color='#3fb950')
ax.axhline(0, color='white', lw=1, ls='--')
ax.set_xlabel('Fitted values (log scale)')
ax.set_ylabel('Residuals (log scale)')
ax.set_title('Residuals vs Fitted')

# 2. Q-Q plot of residuals
ax = axes[0, 1]
(osm, osr), (slope, intercept, r_qq) = stats.probplot(residuals_log, dist='norm')
ax.scatter(osm, osr, color='#3fb950', s=10, alpha=0.5)
xl = np.linspace(min(osm), max(osm), 100)
ax.plot(xl, slope * xl + intercept, color='#ff7b72', lw=2)
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title(f'Q-Q Plot (r={r_qq:.3f})')

# 3. Histogram of residuals
ax = axes[1, 0]
ax.hist(residuals_log, bins=50, color='#3fb950', alpha=0.7, edgecolor='none', density=True)
x_range = np.linspace(residuals_log.min(), residuals_log.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, residuals_log.mean(), residuals_log.std()),
        color='#ff7b72', lw=2, label='Normal fit')
ax.set_xlabel('Residuals (log scale)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend()

# 4. Actual vs Predicted (original scale)
ax = axes[1, 1]
ax.scatter(y_test_original, y_pred_orig, alpha=0.3, s=12, color='#58a6ff')
lim = max(y_test_original.max(), y_pred_orig.max())
ax.plot([0, lim], [0, lim], 'w--', lw=1.5, alpha=0.6, label='Perfect prediction')
ax.set_xlabel('Actual duration (min)')
ax.set_ylabel('Predicted duration (min)')
ax.set_title('Actual vs Predicted (original scale)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Error distribution by duration quantile ──────────────────────────
error_df = pd.DataFrame({
    'actual': y_test_original,
    'predicted': y_pred_orig,
    'abs_error': np.abs(y_test_original - y_pred_orig),
})
error_df['dur_bin'] = pd.qcut(error_df['actual'], q=4,
                               labels=['Q1 (fast)', 'Q2', 'Q3', 'Q4 (slow)'])

fig, ax = plt.subplots(figsize=(10, 5))
error_df.boxplot(column='abs_error', by='dur_bin', ax=ax,
                  patch_artist=True,
                  boxprops=dict(facecolor='#58a6ff', alpha=0.7),
                  medianprops=dict(color='white', lw=2))
ax.set_xlabel('Duration Quartile')
ax.set_ylabel('Absolute Error (min)')
ax.set_title('Prediction Error by Duration Quartile')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('Error stats by quartile:')
print(error_df.groupby('dur_bin')['abs_error'].describe().round(4))

---
## 10 · Hyperparameter Tuning Tips for Random Forest

### Key Hyperparameters (ordered by impact)

#### `n_estimators` (Number of Trees)
- More trees → lower variance, diminishing returns past ~300–500.
- **Never hurts to increase** (only cost is compute time).
- Start with 300, scale to 500–1000 for production.
- Use `warm_start=True` to incrementally add trees and monitor OOB score.

#### `max_depth` (Maximum Tree Depth)
- Controls tree complexity. `None` = fully grown trees (default).
- **Reduce to prevent overfitting**: try `[8, 12, 16, 20, None]`.
- Shallow trees (depth 5–8) → high bias, low variance (good for small datasets).
- Deep trees → low bias, high variance (risk of overfitting).

#### `min_samples_leaf` (Minimum Samples at Leaf)
- Prevents splits that create leaves with too few samples.
- **Increase to reduce overfitting**: try `[1, 2, 4, 8, 16]`.
- Higher values → smoother predictions.

#### `min_samples_split` (Minimum Samples to Split a Node)
- Similar to `min_samples_leaf` but applied at internal nodes.
- Try `[2, 5, 10, 20]`.

#### `max_features` (Features Considered per Split)
- Controls randomness/decorrelation between trees.
- `'sqrt'` — good default for regression.
- Lower values → more decorrelated trees → better ensemble performance.
- Try `['sqrt', 'log2', 0.3, 0.5, 0.7]`.

#### `bootstrap` (Sampling with Replacement)
- `True` (default) — each tree sees a bootstrap sample → enables OOB score.
- `False` — each tree sees the full dataset → can slightly reduce bias.

### General Tips
- **No feature scaling needed** — tree-based models are scale-invariant.
- **Label encoding** suffices for categoricals (no need for one-hot).
- Monitor **train vs test R²** gap — large gap indicates overfitting.
- Use **OOB score** (`oob_score=True`) as a quick validation proxy.
- Use `RandomizedSearchCV` over `GridSearchCV` — more efficient for large spaces.
- **Permutation importance** is more reliable than MDI for correlated features.
- Consider **max_samples** (fraction of data per tree) as additional regularisation.
- For **production**, use `n_jobs=-1` and consider `joblib` for model serialisation.

In [ ]:
print('═══ RANDOM FOREST REGRESSION SUMMARY ═══════════════════════════════════')
print()
print(f'Best model: RF Tuned')
print(f'  R² (log scale):      {metrics_tuned["R² (log)"]}')
print(f'  Adj R² (log scale):  {metrics_tuned["Adj R² (log)"]}')
print(f'  R² (original scale): {metrics_tuned["R² (original)"]}')
print(f'  MAE:                 {metrics_tuned["MAE (min)"]} min')
print(f'  RMSE:                {metrics_tuned["RMSE (min)"]} min')
print(f'  MAPE:                {metrics_tuned["MAPE (%)"]}%')
print()
print(f'Best hyperparameters: {random_search.best_params_}')
print(f'Features used: {X.shape[1]}')
print()
print('Top 5 features (MDI):')
for feat, imp in mdi_imp.head(5).items():
    print(f'  {feat:<35} {imp:.4f}')
print('\n✓ Random Forest Regression notebook complete.')

In [ ]:
import joblib
from pathlib import Path

OUTPUT_DIR = Path('/content/drive/MyDrive/dataset')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / 'rf_model.joblib'

# Export bundle matching engine.py expected format
model_bundle = {
    'model'        : rf_tuned,
    'model_type'   : 'random_forest',
    'feature_names': list(X.columns),
    'os_encoding'  : os_median_enc.to_dict(),
    'os_global'    : float(OS_GLOBAL),
    'lang_encoding': lang_mean_enc.to_dict(),
    'lang_global'  : float(LANG_GLOBAL),
    'log_target'   : True,
}

joblib.dump(model_bundle, model_path)
print(f'Model bundle saved → {model_path}')
print(f'Features ({len(model_bundle["feature_names"])}): {model_bundle["feature_names"]}')

# Verify load
loaded = joblib.load(model_path)
probe = loaded['model'].predict(X_test.head(3))
print(f'Load verification — log-space: {probe}')
print(f'  → original scale (expm1): {np.expm1(probe)} min')
